
## Load processed datasets

In this step, we load the training and dataset generated by the data ingestion notebook.These datasets are stored as serialized pickle files to ensure reproducibility and a clear separation between data ingestion and preprocessing.

No transformations are applied at this stage. 

The dataset contains:
- Sensor measurements (IMU, thermopile, and TOF)
- Metadata (sequence identifiers, subject information, etc.)
- Target labels (gesture)

We verify:
- Dataset shape
- Number of unique sequences
- Minimum sequence length

This ensures the dataset is intact before preprocessing.


In [1]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PROCESSED = Path("../data/processed")

with open(DATA_PROCESSED / "train.pkl", "rb") as f:
    train_df = pickle.load(f)


print("Train:", train_df.shape)

print("Unique sequences:", train_df["sequence_id"].nunique())
print("Min seq length:", train_df.groupby("sequence_id").size().min())

Train: (574945, 341)
Unique sequences: 8151
Min seq length: 29


## 2. Define Sensor Feature Columns

The dataset contains metadata columns and sensor measurement columns.

For model training, only sensor measurements should be used as input features.Metadata such as sequence identifiers and subject information must be excluded to 
prevent data leakage.


We explicitly define:
- IMU columns (7 features)
- Thermopile columns (5 features)
- TOF columns (320 features)

Total expected sensor features: 332

In [2]:
imu_cols = ["acc_x","acc_y","acc_z","rot_w","rot_x","rot_y","rot_z"]
thm_cols = [f"thm_{i}" for i in range(1,6)]
tof_cols = [c for c in train_df.columns if c.startswith("tof_")]

feature_cols = imu_cols + thm_cols + tof_cols
assert len(feature_cols) == 332

feature_map = {
    "label_col": "gesture",
    "imu_cols": imu_cols,
    "thm_cols": thm_cols,
    "tof_cols": tof_cols,
    "feature_cols": feature_cols
}

print("Total features:", len(feature_cols))


Total features: 332


## 3. Remove Short Sequences

Following the methodology described in the paper, sequences whose lengths are smaller than 
(one standard deviation below the mean sequence length) are removed.

This step ensures consistency in temporal modeling and removes unusually short sequences 
that may not provide sufficient signal for learning.

We compute:
- Mean sequence length
- Standard deviation
- Threshold = mean - std


Sequences below this threshold are excluded from further analysis.

In [3]:
seq_len = train_df.groupby("sequence_id").size()
threshold = seq_len.mean() - seq_len.std()

short_ids = seq_len[seq_len < threshold].index
train_df = train_df[~train_df["sequence_id"].isin(short_ids)].copy()

print("Threshold:", threshold)
print("Remaining sequences:", train_df["sequence_id"].nunique())
print("New min seq length:", train_df.groupby("sequence_id").size().min())


Threshold: 35.14686507955627
Remaining sequences: 8144
New min seq length: 36


## 4. Train-Test Split (Sequence-Level)

To prevent data leakage, the train-test split is performed at the sequence level, 
not at the row level.

This ensures that all timesteps belonging to a given sequence are contained entirely 
within either the training or testing set.

We use:
- 80% of sequences for training
- 20% of sequences for testing
- Random seed = 42 for reproducibility

In [4]:
SEED = 42
TEST_SIZE = 0.2

seq_ids = train_df["sequence_id"].unique()
train_ids, test_ids = train_test_split(seq_ids, test_size=TEST_SIZE, random_state=SEED, shuffle=True)

train_clean = train_df[train_df["sequence_id"].isin(train_ids)].copy()
test_clean  = train_df[train_df["sequence_id"].isin(test_ids)].copy()

assert set(train_clean["sequence_id"]).isdisjoint(set(test_clean["sequence_id"]))
print("Train sequences:", train_clean["sequence_id"].nunique())
print("Test sequences :", test_clean["sequence_id"].nunique())


Train sequences: 6515
Test sequences : 1629


## 5. Save Cleaned Datasets

The cleaned and filtered datasets are saved for downstream model implementation.

The following artifacts are stored:
- `train_clean.pkl`
- `test_clean.pkl`
- `feature_map.pkl`

These files will be used directly in the model implementation notebook.


In [5]:
with open(DATA_PROCESSED / "train_clean.pkl", "wb") as f:
    pickle.dump(train_clean, f)

with open(DATA_PROCESSED / "test_clean.pkl", "wb") as f:
    pickle.dump(test_clean, f)

with open(DATA_PROCESSED / "feature_map.pkl", "wb") as f:
    pickle.dump(feature_map, f)

print("Saved train_clean.pkl, test_clean.pkl, feature_map.pkl")


Saved train_clean.pkl, test_clean.pkl, feature_map.pkl
